# Notebook 2: Data Engineering
## CSV → Parquet Conversion, Schema Validation, Partitioning

**Key Operations**:
1. Load raw CSV (~18GB) using Spark
2. Validate schema and data quality
3. Convert to Parquet (columnar, compressed)
4. Partition by subreddit for optimized queries
5. Create virality label

In [ ]:
# ============================================================================
# Cell 1: Setup
# ============================================================================
import os, sys, time
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import *

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# ============================================================================
# Cell 2: Create Spark Session with Optimized Config
# ============================================================================
spark = (
    SparkSession.builder
    .appName('RedditVirality_DataEngineering')
    .master('local[*]')
    .config('spark.driver.memory', '8g')
    .config('spark.executor.memory', '4g')
    .config('spark.driver.maxResultSize', '4g')
    .config('spark.sql.shuffle.partitions', 200)
    .config('spark.default.parallelism', 8)
    .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer')
    .config('spark.sql.adaptive.enabled', 'true')
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true')
    .config('spark.memory.fraction', 0.8)
    .config('spark.sql.parquet.compression.codec', 'snappy')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f'Spark version: {spark.version}')
print(f'Cores available: {spark.sparkContext.defaultParallelism}')

## Step 1: Data Ingestion

In [ ]:
# ============================================================================
# Cell 3: Load Raw CSV with Schema Validation
# ============================================================================
RAW_CSV = os.path.join(PROJECT_ROOT, 'data', 'raw', 'reddit_posts.csv')

# Define expected schema
expected_schema = StructType([
    StructField('author', StringType(), True),
    StructField('body', StringType(), True),
    StructField('normalizedBody', StringType(), True),
    StructField('subreddit', StringType(), True),
    StructField('subreddit_id', StringType(), True),
    StructField('id', StringType(), True),
    StructField('content', StringType(), True),
    StructField('summary', StringType(), True),
])

start_time = time.time()

# Load CSV with explicit schema for reliability
df_raw = spark.read.csv(
    RAW_CSV,
    header=True,
    schema=expected_schema,
    multiLine=True,
    escape='"',
    quote='"'
)

load_time = time.time() - start_time
print(f'CSV loaded in {load_time:.1f} seconds')
print(f'Partitions: {df_raw.rdd.getNumPartitions()}')
df_raw.printSchema()

In [ ]:
# ============================================================================
# Cell 4: Schema Validation
# ============================================================================
print('SCHEMA VALIDATION')
print('=' * 60)

actual_cols = set(df_raw.columns)
expected_cols = {'author', 'body', 'normalizedBody', 'subreddit', 'subreddit_id', 'id', 'content', 'summary'}

missing = expected_cols - actual_cols
extra = actual_cols - expected_cols

if not missing:
    print('✓ All expected columns present')
else:
    print(f'✗ Missing columns: {missing}')

if extra:
    print(f'  Extra columns: {extra}')

row_count = df_raw.count()
print(f'\nTotal rows: {row_count:,}')
print(f'Total columns: {len(df_raw.columns)}')

In [ ]:
# ============================================================================
# Cell 5: Null Check & Data Quality
# ============================================================================
print('NULL VALUE CHECK')
print('=' * 60)

for col_name in df_raw.columns:
    null_ct = df_raw.filter(F.col(col_name).isNull()).count()
    pct = (null_ct / row_count) * 100
    status = '✓' if pct < 5 else '⚠'
    print(f'  {status} {col_name:20s}: {null_ct:>10,} nulls ({pct:.2f}%)')

# Check for duplicates on 'id'
distinct_ids = df_raw.select('id').distinct().count()
dup_count = row_count - distinct_ids
print(f'\nDuplicate IDs: {dup_count:,}')

## Step 2: Data Cleaning & Label Creation

In [ ]:
# ============================================================================
# Cell 6: Clean Data & Create Virality Label
# ============================================================================
print('DATA CLEANING')
print('=' * 60)

# Drop rows with null body or subreddit (essential columns)
df_clean = df_raw.dropna(subset=['body', 'subreddit', 'normalizedBody'])
clean_count = df_clean.count()
print(f'Rows after dropping nulls in body/subreddit: {clean_count:,} (removed {row_count - clean_count:,})')

# Drop duplicates by ID
df_clean = df_clean.dropDuplicates(['id'])
dedup_count = df_clean.count()
print(f'Rows after deduplication: {dedup_count:,} (removed {clean_count - dedup_count:,})')

# Create virality label based on subreddit activity
subreddit_counts = df_clean.groupBy('subreddit').count()
threshold_row = subreddit_counts.approxQuantile('count', [0.80], 0.01)[0]
print(f'\nVirality threshold (80th percentile): {threshold_row:.0f} posts')

viral_subs = subreddit_counts.filter(F.col('count') >= threshold_row).select('subreddit')
viral_list = [row.subreddit for row in viral_subs.collect()]

df_labeled = df_clean.withColumn(
    'is_viral',
    F.when(F.col('subreddit').isin(viral_list), 1).otherwise(0)
)

# Show class distribution
print('\nClass Distribution:')
df_labeled.groupBy('is_viral').count().show()

## Step 3: Convert to Parquet Format

**Why Parquet?**
- **Columnar storage**: Read only needed columns (faster queries)
- **Compression**: Snappy compression reduces file size ~60-80%
- **Schema preservation**: Data types embedded in file
- **Predicate pushdown**: Filter at storage level, not in-memory
- **Spark-optimized**: Native Parquet reader in Spark is highly efficient

In [ ]:
# ============================================================================
# Cell 7: Write to Parquet (Non-Partitioned)
# ============================================================================
PARQUET_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed', 'reddit_posts.parquet')
os.makedirs(os.path.dirname(PARQUET_DIR), exist_ok=True)

start_time = time.time()

df_labeled.write.mode('overwrite') \
    .option('compression', 'snappy') \
    .parquet(PARQUET_DIR)

write_time = time.time() - start_time
print(f'Parquet written in {write_time:.1f} seconds')
print(f'Location: {PARQUET_DIR}')

# Compare file sizes
import glob
csv_size = os.path.getsize(RAW_CSV) / (1024**3)

parquet_files = glob.glob(os.path.join(PARQUET_DIR, '**', '*.parquet'), recursive=True)
parquet_size = sum(os.path.getsize(f) for f in parquet_files) / (1024**3)

print(f'\nSize Comparison:')
print(f'  CSV:     {csv_size:.2f} GB')
print(f'  Parquet: {parquet_size:.2f} GB')
print(f'  Compression ratio: {csv_size / max(parquet_size, 0.001):.1f}x')

## Step 4: Partition Strategy

**Partitioning by `is_viral` label**:
- Enables partition pruning (Spark reads only relevant partition)
- Reduces I/O when filtering by class
- Improves training data access patterns
- Balanced partitions since label is binary

In [ ]:
# ============================================================================
# Cell 8: Write Partitioned Parquet
# ============================================================================
PARTITIONED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed', 'reddit_partitioned.parquet')

start_time = time.time()

df_labeled.write.mode('overwrite') \
    .partitionBy('is_viral') \
    .option('compression', 'snappy') \
    .parquet(PARTITIONED_DIR)

partition_time = time.time() - start_time
print(f'Partitioned Parquet written in {partition_time:.1f} seconds')
print(f'\nPartition structure:')
for item in sorted(os.listdir(PARTITIONED_DIR)):
    full_path = os.path.join(PARTITIONED_DIR, item)
    if os.path.isdir(full_path):
        inner_files = os.listdir(full_path)
        total_size = sum(os.path.getsize(os.path.join(full_path, f)) 
                        for f in inner_files if f.endswith('.parquet'))
        print(f'  {item}/ -> {len(inner_files)} files, {total_size/(1024**2):.0f} MB')

In [ ]:
# ============================================================================
# Cell 9: Verify Parquet Read Performance
# ============================================================================
print('READ PERFORMANCE COMPARISON')
print('=' * 60)

# Read CSV timing
start = time.time()
df_csv_test = spark.read.csv(RAW_CSV, header=True, inferSchema=True, multiLine=True, escape='"')
_ = df_csv_test.count()
csv_read_time = time.time() - start

# Read Parquet timing
start = time.time()
df_parquet_test = spark.read.parquet(PARQUET_DIR)
_ = df_parquet_test.count()
parquet_read_time = time.time() - start

# Read Partitioned Parquet with filter
start = time.time()
df_part_test = spark.read.parquet(PARTITIONED_DIR).filter(F.col('is_viral') == 1)
_ = df_part_test.count()
partitioned_read_time = time.time() - start

print(f'CSV read time:                {csv_read_time:.1f}s')
print(f'Parquet read time:            {parquet_read_time:.1f}s ({csv_read_time/max(parquet_read_time,0.1):.1f}x faster)')
print(f'Partitioned read (filtered):  {partitioned_read_time:.1f}s ({csv_read_time/max(partitioned_read_time,0.1):.1f}x faster)')

In [ ]:
# ============================================================================
# Cell 10: Create Smaller Sample for Development
# ============================================================================
SAMPLE_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed', 'sample')
os.makedirs(SAMPLE_DIR, exist_ok=True)

# 10% stratified sample
df_sample = df_labeled.sampleBy('is_viral', fractions={0: 0.1, 1: 0.1}, seed=42)
sample_count = df_sample.count()
print(f'Sample size: {sample_count:,} rows (10% of full dataset)')
print('Sample class distribution:')
df_sample.groupBy('is_viral').count().show()

df_sample.write.mode('overwrite') \
    .option('compression', 'snappy') \
    .parquet(os.path.join(SAMPLE_DIR, 'reddit_sample.parquet'))

print(f'Sample saved to {SAMPLE_DIR}')

In [ ]:
# ============================================================================
# Cell 11: Data Engineering Summary
# ============================================================================
print('=' * 60)
print('DATA ENGINEERING SUMMARY')
print('=' * 60)
print(f'Raw CSV:                {csv_size:.2f} GB')
print(f'Parquet (compressed):   {parquet_size:.2f} GB')
print(f'Compression ratio:      {csv_size / max(parquet_size, 0.001):.1f}x')
print(f'Rows (cleaned):         {dedup_count:,}')
print(f'Partitions:             is_viral=0, is_viral=1')
print(f'Sample (10%):           {sample_count:,} rows')
print(f'CSV read:               {csv_read_time:.1f}s')
print(f'Parquet read:           {parquet_read_time:.1f}s')
print(f'Partitioned read:       {partitioned_read_time:.1f}s')
print('=' * 60)

spark.stop()
print('Spark session stopped.')